# Week 5: Can 90 photos teach a computer to see? 📸

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GDG-TUM/AI-ML-Track-GDG-on-Campus-TUM/blob/main/weekly-sessions/week-05-deep-learning-in-practice/photo_classifier.ipynb)

**GDG on Campus TUM · AI/ML Track**

Week 2's reader learned from 1,347 tidy 8 × 8 digits. Your photos are 224 × 224 pixels, in colour, with messy backgrounds, and you only have about 30 of each kind. **Can that possibly be enough?**

Today you use a real deep-learning framework (**Keras**), a free GPU, and a model that someone else already trained.

### How this notebook is built

| Part | You learn | Built on |
|:-:|---|---|
| 1 | **Week 4 in Keras:** the same 65-knob network in a few lines | Week 4 |
| 2 | **Your photos as numbers:** `X` and `y`, then two piles | Weeks 2 and 3 |
| 3 | **What a convolution sees:** a filter, slid across a photo by hand | Week 2: pictures are numbers |
| 4 | **A CNN from scratch:** it overfits on 90 photos | Week 3: training vs validation |
| 5 | **Transfer learning:** borrow a model that has seen 1.4 million photos | Part 4's problem |
| 6 | **Mistakes:** confusion matrix and wrong photos | Week 2 |
| 7 | **When not to trust it:** the same photos in different light | Weeks 1, 2 and 4 |

### Before you start: turn on the GPU

In Colab: **Runtime → Change runtime type → T4 GPU → Save**. Everything also runs without a GPU, only slower (a few minutes). Please switch the GPU off when you're done, because free capacity is shared.

*Running locally instead? Run `pip install keras jax[cpu] pillow` first.*

In [ ]:
import keras
import matplotlib.pyplot as plt
import numpy as np

print("Keras", keras.__version__, "running on", keras.backend.backend())

---
# Part 1: Week 4 in Keras

Last week you wrote the moons network by hand: 65 knobs, tanh, sigmoid, cross-entropy, backprop and gradient descent. Here is the **same network** in Keras.

In [ ]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

X, y = make_moons(n_samples=500, noise=0.2, random_state=0)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)

keras.utils.set_random_seed(0)  # repeatable results, like week 1's seed
model = keras.Sequential(
    [
        keras.Input(shape=(2,)),  # 2 inputs: x1 and x2
        keras.layers.Dense(16, activation="tanh"),  # week 4's hidden layer
        keras.layers.Dense(1, activation="sigmoid"),  # week 4's output neuron
    ]
)
model.summary()

**65 parameters**: the same 65 knobs as last week. A `Dense` layer is a set of neurons that each take a weighted sum of *every* input.

Now we train it. `compile` picks the loss and how to walk downhill; `fit` runs the walk.

In [ ]:
model.compile(
    optimizer=keras.optimizers.SGD(learning_rate=0.5),  # SGD = gradient descent
    loss="binary_crossentropy",  # week 4's cross-entropy
    metrics=["accuracy"],
)
model.fit(X_train, y_train, epochs=100, batch_size=32, verbose=0)  # verbose=0: train quietly

loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"test accuracy: {accuracy:.1%}")

About the same accuracy as your NumPy network. **Keras is not magic: `fit` is your week 4 loop**, with backprop worked out automatically.

| Week 4, by hand | Keras |
|---|---|
| `W1`, `b1`, `np.tanh` | `Dense(16, activation="tanh")` |
| `cross_entropy` | `loss="binary_crossentropy"` |
| `knobs - learning_rate * slopes` | `optimizer=SGD(learning_rate=...)` |
| `backprop_slopes` | done for you |
| The training loop | `fit` |

Two new words: Keras steps downhill after every **mini-batch** of 32 examples, rather than after all 375. One pass through all the training data is an **epoch**. Many small steps usually learn faster than a few big ones.

---
# Part 2: Your photos as numbers

## 2.1 Upload your photos

1. On your computer, make a folder called `my_photos`, with **one sub-folder per kind of object**, for example `my_photos/mugs`, `my_photos/shoes`, `my_photos/keys`. Put about 30 photos in each. JPG or PNG; iPhones may save HEIC, so set *Camera → Formats → Most Compatible* or convert them.
2. Zip it to `my_photos.zip`, open the **📁 Files** panel on the left in Colab, and drag the zip in.
3. Run the next cell.

**No photos?** The cell uses a sample instead: 3 kinds of flowers, 30 photos each, from TensorFlow's flowers dataset (CC BY 2.0; the photographers are credited in its `LICENSE.txt`).

In [ ]:
import shutil
from pathlib import Path

PHOTO_FOLDER = Path("my_photos")

if Path("my_photos.zip").exists() and not PHOTO_FOLDER.exists():
    shutil.unpack_archive("my_photos.zip", ".")  # unzip your upload

if not PHOTO_FOLDER.exists():
    print("No my_photos folder, so we use a sample: 3 kinds of flowers, 30 photos each.")
    download = keras.utils.get_file(
        "flower_photos",
        "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz",
        untar=True,
    )
    flowers = next(Path(download).rglob("daisy")).parent  # the folder that holds the flower kinds
    for kind in ["daisy", "sunflowers", "tulips"]:
        (PHOTO_FOLDER / kind).mkdir(parents=True)
        for photo in sorted((flowers / kind).glob("*.jpg"))[:30]:
            shutil.copy(photo, PHOTO_FOLDER / kind)

class_names = sorted(folder.name for folder in PHOTO_FOLDER.iterdir() if folder.is_dir())
print("classes:", class_names)

## 2.2 Photos to arrays

Same as week 2: every picture becomes an array of numbers. Each photo is resized to 224 × 224 pixels with 3 colours (red, green, blue), so one photo is 224 × 224 × 3 numbers from 0 to 255. The label is the position of its folder in `class_names`.

In [ ]:
images = []
labels = []
for label, name in enumerate(class_names):
    for file in sorted((PHOTO_FOLDER / name).iterdir()):
        if file.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
            continue  # skip anything that isn't a photo
        picture = keras.utils.load_img(file, target_size=(224, 224))  # open and resize
        images.append(keras.utils.img_to_array(picture))  # picture -> array of numbers
        labels.append(label)

X = np.array(images)
y = np.array(labels)
print("X:", X.shape, "-> (photos, height, width, colours)")
print("y:", y.shape, "| photos per class:", np.bincount(y))

In [ ]:
fig, axes = plt.subplots(1, 6, figsize=(14, 2.8))
spread_out = np.linspace(0, len(X) - 1, 6).astype(int)  # 6 photos from across all classes
for ax, i in zip(axes, spread_out, strict=True):
    ax.imshow(X[i].astype("uint8"))
    ax.set_title(class_names[y[i]])
    ax.axis("off")
plt.show()

## 2.3 Two piles

As in week 3, we keep a **validation** pile to judge the model. With so few photos we use two piles here. Your real test set will be **new photos you take later**, in the take-home challenge.

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=0
)  # stratify: the same mix of classes in both piles
print("training photos:", len(X_train), "| validation photos:", len(X_val))

---
# Part 3: What a convolution sees

Week 2 compared whole pictures pixel by pixel, and a one-pixel shift broke it. Photos are worse: the object can be anywhere, at any size.

A **convolution** works differently. It slides a small **filter**, here 3 × 3 numbers, across the whole image, looking for one small pattern **everywhere**: an edge, a corner, a patch of colour. The filter below lights up where the image goes from dark (left) to bright (right), a **vertical edge**.

In [ ]:
gray = X[0].mean(axis=2)  # average the 3 colours -> a grayscale image
size = gray.shape[0] - 2  # the output is 2 pixels smaller

vertical = np.array([[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]])  # dark on the left, bright on the right
horizontal = vertical.T  # dark on top, bright below


def convolve(image, edge_filter):
    out = np.zeros((size, size))
    for row in range(3):
        for col in range(3):  # slide: add up each filter number × the matching shifted image
            out += edge_filter[row, col] * image[row : row + size, col : col + size]
    return np.abs(out)


fig, axes = plt.subplots(1, 3, figsize=(11, 3.8))
axes[0].imshow(X[0].astype("uint8"))
axes[0].set_title("photo")
axes[1].imshow(convolve(gray, vertical), cmap="gray")
axes[1].set_title("vertical-edge filter")
axes[2].imshow(convolve(gray, horizontal), cmap="gray")
axes[2].set_title("horizontal-edge filter")
for ax in axes:
    ax.axis("off")
plt.show()

Each filter finds its own pattern wherever it appears in the photo. A **convolutional neural network (CNN)** stacks layers of filters, and **learns** the filter numbers by gradient descent instead of us choosing them. Early layers learn edges, the next ones combine edges into textures and shapes, and later ones respond to petals, handles or wheels.

---
# Part 4: A CNN from scratch

A small CNN: two layers of learned filters (`Conv2D`), each followed by **pooling**, which shrinks the image by keeping the strongest response in each 4 × 4 patch. A final `Dense` layer then picks the class.

New for more than two classes: **softmax** is sigmoid's big sibling, giving one probability per class that add up to 1. The loss `sparse_categorical_crossentropy` is cross-entropy for many classes. **Adam** is a smarter gradient descent that adjusts the step size for each knob.

🤔 **Predict first:** after 15 epochs, will training and validation accuracy be close?

In [ ]:
keras.utils.set_random_seed(0)
scratch_cnn = keras.Sequential(
    [
        keras.Input(shape=(224, 224, 3)),
        keras.layers.Rescaling(1 / 255),  # 0-255 -> 0-1
        keras.layers.Conv2D(16, 3, activation="relu"),  # 16 learned 3 × 3 filters
        keras.layers.MaxPooling2D(4),  # shrink: keep the strongest response per 4 × 4 patch
        keras.layers.Conv2D(32, 3, activation="relu"),  # 32 filters built from the first 16
        keras.layers.MaxPooling2D(4),
        keras.layers.Flatten(),  # everything into one long row
        keras.layers.Dense(len(class_names), activation="softmax"),  # one probability per class
    ]
)
scratch_cnn.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
scratch_history = scratch_cnn.fit(
    X_train, y_train, validation_data=(X_val, y_val), epochs=15, verbose=0
)
print(f"knobs: {scratch_cnn.count_params():,}")

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(scratch_history.history["accuracy"], color="#EA4335", label="training")
plt.plot(scratch_history.history["val_accuracy"], color="#4285F4", label="validation")
plt.xlabel("epoch")
plt.ylabel("accuracy")
plt.title("From scratch: training soars, validation lags")
plt.legend()
plt.show()

Week 3's overfitting picture. The network memorises the training photos but hasn't learned what the objects really look like: **63 photos are nowhere near enough to learn vision from scratch.**

---
# Part 5: Transfer learning

## 5.1 Borrow a model that can already see

**MobileNetV3** was trained by Google on **ImageNet**: 1.4 million photos in 1,000 categories. Its filters already detect edges, textures and shapes. Let's ask it what it sees in one of your photos. (It only knows its own 1,000 categories, so its labels may be close rather than exact.)

In [ ]:
imagenet_model = keras.applications.MobileNetV3Small(input_shape=(224, 224, 3))  # ~10 MB download
probabilities = imagenet_model.predict(X[:1], verbose=0)
top_3 = keras.applications.imagenet_utils.decode_predictions(probabilities, top=3)[0]
for _, name, probability in top_3:
    print(f"{name:20s} {probability:.0%}")

## 5.2 Keep its eyes, give it a new last layer

We take MobileNetV3 **without** its final layer, **freeze** its knobs so they don't change, and add one new `Dense` layer for *your* classes. Only that layer is trained.

In [ ]:
base = keras.applications.MobileNetV3Small(
    input_shape=(224, 224, 3), include_top=False, weights="imagenet", pooling="avg"
)  # include_top=False: without its 1,000-category last layer
base.trainable = False  # freeze: keep the borrowed knobs as they are

keras.utils.set_random_seed(0)
transfer_model = keras.Sequential(
    [
        keras.Input(shape=(224, 224, 3)),
        base,  # the borrowed eyes
        keras.layers.Dense(len(class_names), activation="softmax"),  # our new last layer
    ]
)
trainable = sum(int(np.prod(weight.shape)) for weight in transfer_model.trainable_weights)
print(f"borrowed knobs (frozen): {base.count_params():,}")
print(f"knobs we train         : {trainable:,}")

In [ ]:
transfer_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.003),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
transfer_history = transfer_model.fit(
    X_train, y_train, validation_data=(X_val, y_val), epochs=15, batch_size=16, verbose=0
)

plt.figure(figsize=(7, 4))
plt.plot(scratch_history.history["val_accuracy"], color="#9AA0A6", label="from scratch")
plt.plot(transfer_history.history["val_accuracy"], color="#34A853", label="transfer learning")
plt.xlabel("epoch")
plt.ylabel("validation accuracy")
plt.title("Same photos, borrowed eyes")
plt.legend()
plt.show()

Far better, from the same few photos, by training only a few thousand knobs. That is **transfer learning**, and it's how most real image, text and speech models are built. Even large language models (Phase 2) are a pretrained model adapted to a task.

---
# Part 6: Where does it go wrong?

Week 2's tools: a confusion matrix, then the wrong photos themselves.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

guesses = transfer_model.predict(X_val, verbose=0).argmax(axis=1)  # the most likely class
print(f"validation accuracy: {(guesses == y_val).mean():.1%}")

ConfusionMatrixDisplay.from_predictions(
    y_val, guesses, display_labels=class_names, cmap="Blues", colorbar=False
)
plt.show()

In [ ]:
wrong = np.where(guesses != y_val)[0]
print(len(wrong), "mistakes out of", len(y_val))

if len(wrong) > 0:
    fig, axes = plt.subplots(1, len(wrong), figsize=(2.6 * len(wrong), 3), squeeze=False)
    for ax, i in zip(axes[0], wrong, strict=True):
        ax.imshow(X_val[i].astype("uint8"))
        ax.set_title(f"true {class_names[y_val[i]]}\nguess {class_names[guesses[i]]}", fontsize=9)
        ax.axis("off")
    plt.show()

Look closely: is the object small, half-hidden, or on a busy background? Mistakes tell you what photos to collect next.

---
# 🛡️ Part 7: When not to trust it

Your photos were taken on your phone, in your room, in your light. What about photos taken somewhere else? We change the light on the validation photos:

In [ ]:
conditions = {
    "as taken": X_val,
    "dim room": X_val * 0.3,  # every pixel 70% darker
    "warm lamp": X_val * np.array([1.0, 0.75, 0.35]),  # less green and blue
    "black & white": X_val.mean(axis=3, keepdims=True).repeat(3, axis=3),  # no colour
}

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for ax, (name, photos) in zip(axes, conditions.items(), strict=True):
    ax.imshow(photos[0].astype("uint8"))
    ax.set_title(name)
    ax.axis("off")
plt.show()

🤔 **Predict first:** you can still name every object in all four versions. Can the model?

In [ ]:
for name, photos in conditions.items():
    _, accuracy = transfer_model.evaluate(photos, y_val, verbose=0)
    print(f"{name:14s} accuracy {accuracy:.0%}")

Accuracy usually drops, often most in black and white. The model leaned on clues (colour, brightness) that happened to work in *your* photos. Same lesson as the storm day, the shifted digits and the far-away moons: **whatever is missing from the data is missing from the model.**

Discuss with your partner:

* Whose photos might this fail on? (Older phones, night shifts, other countries, people with different objects?)
* We photographed **objects, not people**. Why? Faces are personal data, and pretrained models can carry biases from the photos they learned from.

---
## 📖 New words

| What you did | What it is called |
|---|---|
| Keras: layers, `compile`, `fit` | a deep-learning **framework** |
| Neurons that each see every input | a **Dense** layer |
| Steps after every 32 examples / one pass through all the data | a **mini-batch** / an **epoch** |
| A small pattern-finder slid across an image | a **filter** in a **convolution** |
| Layers of learned filters | a **convolutional neural network (CNN)** |
| One probability per class, adding up to 1 | **softmax** |
| A model trained by someone else on lots of data | a **pretrained** model |
| Keeping its knobs fixed | **freezing** |
| Reusing it, with a new last layer, for your task | **transfer learning** |
| Unfreezing some layers and training them very gently | **fine-tuning** |

---
## 🏁 Take-home challenge

1. **The real test.** Take **5 new photos of each object**, in different light or places. Put them in a `new_photos` folder (same sub-folders), load them as in 2.2, and score the model. This is your honest test set (week 3).
2. **One improvement.** Try one and report whether validation accuracy moved:
   * more photos per class;
   * **data augmentation** (week 2's challenge): add `keras.layers.RandomFlip("horizontal")` and `keras.layers.RandomRotation(0.1)` right after `keras.Input` in Part 5;
   * **fine-tuning:** after Part 5, set `base.trainable = True`, compile again with `Adam(learning_rate=1e-5)`, and train for 5 more epochs.
3. **Finish your mini-project** with your partner, and prepare your 3-minute presentation for week 6.

**Before you leave Colab:** Runtime → Change runtime type → CPU, so the free GPU goes back to the pool.

In [ ]:
# Your code here